# HealthRAG — Etapas 3 y 4: índices y recuperación

Comparación cualitativa de: **básica** (embeddings), **híbrida** (BM25 + embeddings + RRF) y **avanzada** (híbrida + Cross-Encoder).

In [1]:
import os, sys
from pathlib import Path

# Raíz del proyecto: funciona abriendo el notebook desde notebooks/ o desde la raíz.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from app.config import load_config, resolve_path
from app.utils.logging_config import configure_logging

cfg = load_config()
logger = configure_logging(cfg)
print("Proyecto:", ROOT)

Proyecto: HealthRAG


In [2]:
from app.retrieval.retriever import HealthRetriever
retriever = HealthRetriever(cfg)   # construye el índice FAISS si no existe
print('Embeddings:', retriever.embedder.name, '| Reranker:', getattr(retriever.reranker, 'name', None))

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

... (11 outputs left out)

Batches:   0%|          | 0/48 [00:00<?, ?it/s]

2026-11-15 12:45:25,557 | INFO | healthrag.retrieval | Índice vectorial: 1514 chunks, intfloat/multilingual-e5-small, 239.1s


2026-11-15 12:45:25,574 | INFO | healthrag.retrieval | Retriever listo: 1514 chunks (240535 ms)


Embeddings: intfloat/multilingual-e5-small | Reranker: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1


In [3]:
import pandas as pd
query = 'valores de presion arterial para diagnosticar hipertension en consultorio'
rows = []
for mode in ('basic', 'hybrid', 'advanced'):
    res = retriever.retrieve(query, mode=mode, top_k=5)
    for r in res.chunks:
        rows.append({'modo': mode, 'rank': r.rank, 'chunk_id': r.chunk.chunk_id, **{k: round(v, 3) for k, v in r.scores.items()}, 'texto': r.chunk.text[:150]})
    print(mode, {k: round(v, 1) for k, v in res.timings_ms.items()}, 'evidencia débil:', res.weak_evidence)
display(pd.DataFrame(rows))

basic

 {'vector_ms': 59.4, 'total_ms': 59.4} evidencia débil: False
hybrid {'vector_ms': 0.4, 'bm25_ms': 14.1, 'fusion_ms': 0.1, 'total_ms': 14.7} evidencia débil: False


advanced {'vector_ms': 0.4, 'bm25_ms': 3.3, 'fusion_ms': 0.1, 'rerank_ms': 3938.8, 'total_ms': 3942.6} evidencia débil: False


,modo,rank,chunk_id,vector,texto,rrf,bm25,rerank
0,basic,1,GPCN_HTA_p10_c24,0.908,Sospecha de errores diagnósticos Si se sospech...,NaN,NaN,NaN
1,basic,2,GPCN_HTA_p4_c6,0.903,HTA: hipertensión arterial. HVI: hipertrofia v...,NaN,NaN,NaN
2,basic,3,GPCN_HTA_p12_c31,0.903,- En personas con HTA de 80 años y más que ini...,NaN,NaN,NaN
3,basic,4,GPCN_HTA_p11_c27,0.902,Criterios diagnósticos de HTA según el método ...,NaN,NaN,NaN
4,basic,5,GPCN_HTA_p8_c18,0.893,2. Diagnóstico y puntos de corte Se recomienda...,NaN,NaN,NaN
5,hybrid,1,GPCN_HTA_p10_c24,0.908,Sospecha de errores diagnósticos Si se sospech...,0.033,20.325,NaN
6,hybrid,2,GPCN_HTA_p12_c31,0.903,- En personas con HTA de 80 años y más que ini...,0.032,14.352,NaN
7,hybrid,3,GPCN_HTA_p8_c18,0.893,2. Diagnóstico y puntos de corte Se recomienda...,0.032,15.314,NaN
8,hybrid,4,GPCN_HTA_p4_c6,0.903,HTA: hipertensión arterial. HVI: hipertrofia v...,0.031,13.105,NaN
9,hybrid,5,GPCN_HTA_p12_c30,0.889,3. Metas del tratamiento Las metas varían segú...,0.030,14.075,NaN
